<a href="https://colab.research.google.com/github/gouthamsankar-clg/AI-Lab/blob/main/Lab_5/SimulatedAnnealing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import math

def read_state():
    s = input("Enter state (column i = row of queen, e.g. 0 1 2 3 4 5 6 7): ").split()
    state = [int(x) for x in s]
    n = len(state)
    if n < 4 or sorted(state) != list(range(n)):
        raise ValueError(f"Enter a permutation of 0..{n-1} (n >= 4)")
    return state

def read_params():
    s = input("Enter initial_temp cooling_factor max_iterations (Enter for 10 0.99 2000): ").split()
    if not s:
        return 10.0, 0.99, 2000
    return float(s[0]), float(s[1]), int(s[2])

def calculate_cost(board):
    conflicts = 0
    n = len(board)
    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = board[c1]
            r2 = board[c2]
            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflicts += 1
    return conflicts

def get_neighbor(board):
    n = len(board)
    neighbor = list(board)
    col = random.randint(0, n - 1)
    available_rows = [r for r in range(n) if r != board[col]]
    neighbor[col] = random.choice(available_rows)
    return neighbor

def simulated_annealing(initial_board, initial_temp, cooling_factor, max_iterations):
    current_board = list(initial_board)
    current_cost = calculate_cost(current_board)
    T = initial_temp

    print(f"Initial Board: {current_board} | Initial Cost: {current_cost} conflicts\n")
    print("-" * 65)
    print(f"{'Iter':<6} | {'Current Board':<25} | {'Cost':<5} | {'Temp':<7} | {'Action'}")
    print("-" * 65)

    iteration = 0
    while current_cost > 0 and iteration < max_iterations and T > 1e-5:
        iteration += 1
        proposed_board = get_neighbor(current_board)
        proposed_cost = calculate_cost(proposed_board)
        delta_E = proposed_cost - current_cost

        if delta_E < 0:
            current_board = proposed_board
            current_cost = proposed_cost
            action = "Accepted (Better)"
        else:
            P = math.exp(-delta_E / T)
            R = random.random()
            if R < P:
                current_board = proposed_board
                current_cost = proposed_cost
                action = f"Accepted (Worse, P={P:.3f})"
            else:
                action = f"Rejected (Worse, P={P:.3f})"

        if iteration <= 5 or iteration % 100 == 0 or current_cost == 0:
            print(f"{iteration:<6} | {str(current_board):<25} | {current_cost:<5} | {T:<7.3f} | {action}")

        T = T * cooling_factor

    print("-" * 65)
    if current_cost == 0:
        print(f"\nSuccess! Solution Found in {iteration} iterations.")
        print(f"Final Board State: {current_board}")
    else:
        print(f"\nMax iterations reached or system cooled down without finding an optimal solution.")
        print(f"Best Found State: {current_board} with {current_cost} conflicts.")

if __name__ == "__main__":
    start = read_state()
    temp, cooling, iters = read_params()
    simulated_annealing(start, temp, cooling, iters)

Enter state (column i = row of queen, e.g. 0 1 2 3 4 5 6 7): 0 1 2 3 4 5 6 7
Enter initial_temp cooling_factor max_iterations (Enter for 10 0.99 2000): 10 0.99 2000
Initial Board: [0, 1, 2, 3, 4, 5, 6, 7] | Initial Cost: 28 conflicts

-----------------------------------------------------------------
Iter   | Current Board             | Cost  | Temp    | Action
-----------------------------------------------------------------
1      | [0, 1, 2, 3, 4, 5, 6, 2]  | 22    | 10.000  | Accepted (Better)
2      | [0, 4, 2, 3, 4, 5, 6, 2]  | 17    | 9.900   | Accepted (Better)
3      | [0, 4, 2, 3, 4, 5, 3, 2]  | 14    | 9.801   | Accepted (Better)
4      | [0, 4, 2, 3, 4, 5, 3, 6]  | 12    | 9.703   | Accepted (Better)
5      | [0, 4, 4, 3, 4, 5, 3, 6]  | 11    | 9.606   | Accepted (Better)
100    | [6, 3, 2, 7, 2, 2, 2, 5]  | 9     | 3.697   | Rejected (Worse, P=0.444)
200    | [7, 1, 2, 0, 6, 0, 4, 6]  | 4     | 1.353   | Accepted (Better)
300    | [6, 3, 0, 4, 7, 0, 1, 7]  | 4     | 0.495  